In [10]:
import json
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Evidently
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, DataQualityPreset
from evidently.metrics import *

print("✅ Imports OK")

✅ Imports OK


In [11]:
##2 — Chargement des données de référence et de production :

In [13]:
# Chargement des logs de production
logs = []
with open("/Users/mac3/Desktop/AIengineer/projet7/scoring-mlops-deployment/logs/predictions.json", "r") as f:
    for line in f:
        logs.append(json.loads(line))

df_logs = pd.DataFrame(logs)
df_logs["timestamp"] = pd.to_datetime(df_logs["timestamp"])
df_logs = df_logs.sort_values("timestamp")

# Extraction des features depuis les inputs
df_inputs = pd.json_normalize(df_logs["input"])
df_inputs["score"] = df_logs["score"].values
df_inputs["decision"] = df_logs["decision"].values
df_inputs["latence_ms"] = df_logs["latence_ms"].values

print(f"✅ {len(df_inputs)} prédictions chargées")
print(df_inputs.head(3))

✅ 500 prédictions chargées
   EXT_SOURCE_1  EXT_SOURCE_2  EXT_SOURCE_3  AMT_CREDIT  AMT_INCOME_TOTAL  \
0         0.574         0.494         0.949   790549.91          65272.66   
1         0.902         0.443         0.129   909824.07         419797.40   
2         0.250         0.068         0.257   152632.87          30602.14   

   DAYS_BIRTH  DAYS_EMPLOYED     score decision  latence_ms  
0      -21681          -5808  0.319349    REFUS      33.281  
1      -14136          -7993  0.188673    REFUS      10.075  
2      -12354          -2821  0.712178    REFUS      52.672  


In [14]:
#  3 — Découpage référence vs production :

In [15]:
# Découpage : première moitié = référence, deuxième moitié = production
midpoint = len(df_inputs) // 2
df_reference = df_inputs.iloc[:midpoint].reset_index(drop=True)
df_production = df_inputs.iloc[midpoint:].reset_index(drop=True)

# Features numériques uniquement pour Evidently
features = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
            "AMT_CREDIT", "AMT_INCOME_TOTAL", "DAYS_BIRTH",
            "DAYS_EMPLOYED", "score", "latence_ms"]

df_ref_ev = df_reference[features]
df_prod_ev = df_production[features]

print(f"✅ Référence : {len(df_ref_ev)} lignes")
print(f"✅ Production : {len(df_prod_ev)} lignes")
print(df_ref_ev.describe().round(3))

✅ Référence : 250 lignes
✅ Production : 250 lignes
       EXT_SOURCE_1  EXT_SOURCE_2  EXT_SOURCE_3  AMT_CREDIT  AMT_INCOME_TOTAL  \
count       250.000       250.000       250.000     250.000           250.000   
mean          0.522         0.494         0.525  515175.229        261514.716   
std           0.295         0.288         0.297  282883.064        136465.172   
min           0.005         0.002         0.001   51388.940         30602.140   
25%           0.250         0.248         0.227  265244.090        143681.080   
50%           0.513         0.518         0.554  520304.245        257618.400   
75%           0.814         0.734         0.776  745615.530        393255.658   
max           0.997         1.000         0.994  999457.910        499705.580   

       DAYS_BIRTH  DAYS_EMPLOYED    score  latence_ms  
count     250.000        250.000  250.000     250.000  
mean   -15227.028      -4911.568    0.342      29.290  
std      5237.106       2745.600    0.177       7.9

In [16]:
##  4 — Rapport Evidently Data Drift :


In [17]:
# Rapport Evidently — Data Drift
report = Report(metrics=[
    DataDriftPreset(),
    DataQualityPreset(),
])

report.run(reference_data=df_ref_ev, current_data=df_prod_ev)

# Sauvegarde HTML
report.save_html("/Users/mac3/Desktop/AIengineer/projet7/scoring-mlops-deployment/monitoring/drift_report.html")
print("✅ Rapport sauvegardé : monitoring/drift_report.html")


✅ Rapport sauvegardé : monitoring/drift_report.html


In [18]:
print("=" * 60)
print("CONCLUSIONS — ANALYSE DATA DRIFT")
print("=" * 60)

print("""
🟢 RÉSULTAT GLOBAL : Dataset Drift NON détecté
   Seuil Evidently : 50% de features driftées
   Observé : 11.1% (1/9 features)

⚠️  FEATURE DRIFTÉE : EXT_SOURCE_1
   - Référence : moyenne 0.52, médiane 0.51
   - Production : moyenne 0.45, médiane 0.39
   - Baisse de ~13% de la moyenne → clients moins bien notés
   - p-value K-S : 0.015 < 0.05 → drift statistiquement significatif

✅ FEATURES STABLES (8/9) :
   - EXT_SOURCE_2, EXT_SOURCE_3 : scores externes stables
   - DAYS_BIRTH, DAYS_EMPLOYED : profil démographique stable
   - AMT_CREDIT, AMT_INCOME_TOTAL : montants stables
   - score, latence_ms : performances API stables

📋 POINTS DE VIGILANCE :
   1. EXT_SOURCE_1 en baisse → surveiller l'impact sur le taux de refus
   2. Latence stable (~30ms) → pas de dégradation performance
   3. Score moyen légèrement en hausse (0.34 → 0.35) → cohérent avec drift
   4. Recommandation : ré-entraîner si EXT_SOURCE_1 drift s'amplifie

🎯 ACTION RECOMMANDÉE :
   - Alerte si p-value EXT_SOURCE_1 < 0.01 sur fenêtre glissante
   - Monitoring hebdomadaire des distributions
   - Ré-entraînement si 3+ features driftent simultanément
""")

print("=" * 60)

CONCLUSIONS — ANALYSE DATA DRIFT

🟢 RÉSULTAT GLOBAL : Dataset Drift NON détecté
   Seuil Evidently : 50% de features driftées
   Observé : 11.1% (1/9 features)

⚠️  FEATURE DRIFTÉE : EXT_SOURCE_1
   - Référence : moyenne 0.52, médiane 0.51
   - Production : moyenne 0.45, médiane 0.39
   - Baisse de ~13% de la moyenne → clients moins bien notés
   - p-value K-S : 0.015 < 0.05 → drift statistiquement significatif

✅ FEATURES STABLES (8/9) :
   - EXT_SOURCE_2, EXT_SOURCE_3 : scores externes stables
   - DAYS_BIRTH, DAYS_EMPLOYED : profil démographique stable
   - AMT_CREDIT, AMT_INCOME_TOTAL : montants stables
   - score, latence_ms : performances API stables

📋 POINTS DE VIGILANCE :
   1. EXT_SOURCE_1 en baisse → surveiller l'impact sur le taux de refus
   2. Latence stable (~30ms) → pas de dégradation performance
   3. Score moyen légèrement en hausse (0.34 → 0.35) → cohérent avec drift
   4. Recommandation : ré-entraîner si EXT_SOURCE_1 drift s'amplifie

🎯 ACTION RECOMMANDÉE :
   - Ale